# Phase 4 · Diagnostic — does frozen Architecture A carry the outcome signal?

Two full P4_S1 runs ended with identical predictions for every case (train loss = ln 2), so before spending more GPU hours this notebook measures, on a sample, **whether the outcome can be predicted from frozen Phase-1 Architecture A features at all, and from which representation**:
1. Numerical health of the backbone on 512-token chunks: any inf/NaN, fp16 vs fp32 difference, magnitude, and how much features vary **between cases**.
2. Logistic regression (dev-clean Macro-F1) on frozen features: mean / final state / max over the last 512 tokens, the same for the last 128 tokens, and the mean of all chunk-level 128-token windows.
3. A reference ceiling from the text alone: TF-IDF word n-grams + logistic regression on the same cases (a diagnostic only — not a submission).

**Kaggle:** GPU T4 (one is enough) · secret `HF` · internet on · inputs `rohitbatra2004/phase111` and `rohitbatra2004/cs781-phase1-tokenized-corpus`. Time ≈ 15 minutes. Send back the printed output of Cells 8–10 (also saved to `P4_diagnostic/diagnostic.json`).

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and sample sizes.
RUN_NAME = "P4_diagnostic"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
WORK_DIR.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = WORK_DIR
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"
N_TRAIN_SAMPLE = 4000            # balanced random sample of training cases
CHUNK_TOKENS = 510
TAIL_CHARS = 8000                # enough characters for the last 510 tokens
BATCH = 16
USE_AMP = DEVICE.type == "cuda"
print("Outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download Phase_4_CJPE from Hugging Face using the Kaggle secret "HF".
from kaggle_secrets import UserSecretsClient
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_4_CJPE/**"])
P4_DIR = DATA_DIR / "Phase_4_CJPE"
for name in ("Train/train.json", "Train/dev.json", "Test/test.json"):
    assert (P4_DIR / name).is_file(), f"Missing {name}"
print("Phase 4 data ready in", P4_DIR)

In [ ]:
# Cell 4 — Load the splits, lower-case and normalise the text, remove duplicate training cases,
# and build "dev-clean" = dev cases whose ids are NOT in the test set. Test labels are never read.
def normalise(text):
    return " ".join(text.lower().split())

def text_key(text):
    return hashlib.md5(text.encode()).hexdigest()

raw_train = json.load(open(P4_DIR / "Train/train.json"))
raw_dev = json.load(open(P4_DIR / "Train/dev.json"))
raw_test = json.load(open(P4_DIR / "Test/test.json"))

train_cases, seen = [], set()
for d in raw_train:
    text = normalise(d["text"])
    key = text_key(text)
    if key in seen:
        continue
    seen.add(key)
    train_cases.append({"id": str(d["id"]), "text": text, "label": int(d["label"])})

test_cases = [{"id": str(d["id"]), "text": normalise(d["text"])} for d in raw_test]       # labels deliberately ignored
test_ids = {c["id"] for c in test_cases}
dev_all = [{"id": str(d["id"]), "text": normalise(d["text"]), "label": int(d["label"])} for d in raw_dev]
dev_cases = [c for c in dev_all if c["id"] not in test_ids]
del raw_train, raw_dev, raw_test

print(f"train: {len(train_cases):,} unique cases (duplicates removed) | labels {Counter(c['label'] for c in train_cases)}")
print(f"dev  : {len(dev_all):,} cases, of which {len(dev_all) - len(dev_cases):,} are also in test and are excluded "
      f"→ dev-clean {len(dev_cases):,} | labels {Counter(c['label'] for c in dev_cases)}")
print(f"test : {len(test_cases):,} cases")

In [ ]:
# Cell 5 — Balanced training sample + all dev-clean cases; tokenise the end of each case (last 510 tokens).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]"))
rng = random.Random(SEED)
by_label = {0: [c for c in train_cases if c["label"] == 0], 1: [c for c in train_cases if c["label"] == 1]}
sample_train = rng.sample(by_label[0], N_TRAIN_SAMPLE // 2) + rng.sample(by_label[1], N_TRAIN_SAMPLE // 2)
rng.shuffle(sample_train)

def tail_ids(cases):
    enc = tokenizer.encode_batch([c["text"][-TAIL_CHARS:] for c in cases])
    return [[BOS_ID] + (e.ids[1:] if len(c["text"]) > TAIL_CHARS else e.ids)[-CHUNK_TOKENS:] + [EOS_ID] for c, e in zip(cases, enc)]

train_ids, dev_ids = tail_ids(sample_train), tail_ids(dev_cases)
y_train = np.array([c["label"] for c in sample_train]); y_dev = np.array([c["label"] for c in dev_cases])
print(f"train sample {len(train_ids)} (balanced) | dev-clean {len(dev_ids)} | median sequence length "
      f"{int(np.median([len(x) for x in train_ids]))}")
print("End of one case:", tokenizer.decode(train_ids[0][-40:]))

In [ ]:
# Cell 6 — Phase-1 Architecture A classes and the pretrained weights, verified against the Phase-1 metadata.
# The weights stay in memory so that build_model() can create a fresh adapted copy.
a_paths = sorted(glob.glob("/kaggle/input/**/architecture_A/model_A_state_dict.pt", recursive=True))
assert a_paths, "Add the dataset rohitbatra2004/phase111 as an input."
A_ROOT = Path(a_paths[0]).parent
A_METADATA = json.load(open(A_ROOT / "model_A_metadata.json"))
A_CONFIG = A_METADATA["model_config"]
assert A_CONFIG["pre_norm"] and A_CONFIG["tie_embeddings"], "Phase-1 A is a pre-LN decoder with tied embeddings"

class CausalSelfAttention(nn.Module):
    """Same parameters as Phase 1; uses PyTorch's fused causal attention for speed and memory."""
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.n_heads, self.dropout = cfg["n_heads"], cfg["dropout"]
        self.q_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.k_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.v_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.out_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.resid_dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, T, C = x.shape
        split = lambda t: t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        out = F.scaled_dot_product_attention(split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x)),
                                             dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.resid_dropout(self.out_proj(out.transpose(1, 2).reshape(B, T, C)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["use_bias"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["use_bias"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["d_model"])
        self.attention = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["d_model"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        return x + self.feed_forward(self.ln_2(x))

class ArchitectureABackbone(nn.Module):
    """Phase-1 decoder-only LM without the vocabulary projection.
    Returns the final-normalised state of every token: [batch, tokens, d_model]."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(cfg["d_model"])
        # Tied output weight, kept so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_weight = self.token_embedding.weight

    def forward(self, input_ids):
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)

saved = torch.load(a_paths[0], map_location="cpu", weights_only=True)
# The export is either the raw state dict or a dict that wraps it under "state_dict".
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved

check = ArchitectureABackbone(A_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(A_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(A_CONFIG, indent=2))
print(f"Phase-1 Architecture A weights verified from {A_ROOT} — {n_backbone:,} parameters")

In [ ]:
# Cell 7 — Numerical health of the frozen backbone on full 512-token chunks (fp16 vs fp32, inf/NaN, magnitudes).
backbone = ArchitectureABackbone(A_CONFIG); backbone.load_state_dict(PHASE1_STATE, strict=True)
backbone = backbone.to(DEVICE).eval()

def pad(batch):
    T = max(len(x) for x in batch)
    ids = torch.full((len(batch), T), PAD_ID, dtype=torch.long)
    for i, x in enumerate(batch):
        ids[i, :len(x)] = torch.tensor(x)
    return ids.to(DEVICE)

health = {}
with torch.no_grad():
    probe = pad(train_ids[:8])
    s32 = backbone(probe).float()
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        s16 = backbone(probe).float()
    # residual stream before the final LayerNorm, to see whether it overflows fp16
    x = backbone.token_embedding(probe) + backbone.position_embedding(torch.arange(probe.size(1), device=DEVICE))[None]
    for block in backbone.blocks:
        x = block(x)
    health = {"inf_or_nan_fp32": bool(~torch.isfinite(s32).all()), "inf_or_nan_fp16": bool(~torch.isfinite(s16).all()),
              "max_abs_diff_fp16_vs_fp32": float((s16 - s32).abs().max()),
              "final_state_mean_abs": float(s32.abs().mean()),
              "residual_stream_max_abs_before_final_norm": float(x.abs().max()),
              "residual_stream_mean_abs": float(x.abs().mean())}
print(json.dumps(health, indent=2))

In [ ]:
# Cell 8 — Frozen features of each case under several representations (fp32 for safety).
@torch.no_grad()
def features(id_lists):
    out = {k: [] for k in ("mean512", "final512", "max512", "mean_last128", "final_last128", "mean_of_windows128")}
    for s in range(0, len(id_lists), BATCH):
        batch = id_lists[s:s + BATCH]
        ids = pad(batch); mask = ids.ne(PAD_ID); lengths = mask.sum(1)
        st = backbone(ids).float()
        m = mask.unsqueeze(-1).float()
        out["mean512"].append((st * m).sum(1) / m.sum(1))
        out["final512"].append(st[torch.arange(len(batch)), lengths - 1])
        out["max512"].append(st.masked_fill(~mask.unsqueeze(-1), -1e4).max(1).values)
        short = [x[-128:] if len(x) > 128 else x for x in batch]                     # separate pass on the last 128 tokens
        ids2 = pad([[BOS_ID] + x for x in short]); m2 = ids2.ne(PAD_ID)
        st2 = backbone(ids2).float()
        out["mean_last128"].append((st2 * m2.unsqueeze(-1)).sum(1) / m2.sum(1, keepdim=True))
        out["final_last128"].append(st2[torch.arange(len(batch)), m2.sum(1) - 1])
        windows = []                                                                 # 128-token windows encoded separately
        for x in batch:
            body = x[1:-1]
            pieces = [[BOS_ID] + body[i:i + 128] + [EOS_ID] for i in range(0, max(len(body), 1), 128)][:4]
            w = backbone(pad(pieces)).float(); wm = pad(pieces).ne(PAD_ID).unsqueeze(-1).float()
            windows.append(((w * wm).sum(1) / wm.sum(1)).mean(0))
        out["mean_of_windows128"].append(torch.stack(windows))
    return {k: torch.cat(v).cpu().numpy() for k, v in out.items()}

start = time.time()
F_train, F_dev = features(train_ids), features(dev_ids)
print(f"Features extracted in {(time.time() - start) / 60:.1f} min")
for k, v in F_train.items():
    between = v.std(0).mean(); magnitude = np.abs(v).mean()
    print(f"  {k:20s} variation between cases {between:.4f} | magnitude {magnitude:.4f} | ratio {between / magnitude:.3f}")

In [ ]:
# Cell 9 — Logistic regression on each frozen representation: dev-clean Macro-F1 (threshold tuned on dev-clean).
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def best_macro_f1(gold, probs):
    return max(f1_score(gold, (probs >= t).astype(int), average="macro") for t in np.arange(0.2, 0.81, 0.01))

probe_results = {}
for k in F_train:
    scaler = StandardScaler().fit(F_train[k])
    clf = LogisticRegression(max_iter=3000, C=0.5).fit(scaler.transform(F_train[k]), y_train)
    p_dev = clf.predict_proba(scaler.transform(F_dev[k]))[:, 1]
    p_tr = clf.predict_proba(scaler.transform(F_train[k]))[:, 1]
    probe_results[k] = {"train_macro_f1": round(best_macro_f1(y_train, p_tr), 4),
                        "dev_clean_macro_f1": round(best_macro_f1(y_dev, p_dev), 4), "dev_prob_std": round(float(p_dev.std()), 4)}
    print(f"  {k:20s} train {probe_results[k]['train_macro_f1']:.4f} | dev-clean {probe_results[k]['dev_clean_macro_f1']:.4f} "
          f"| spread {probe_results[k]['dev_prob_std']:.3f}")

In [ ]:
# Cell 10 — Text-only reference (TF-IDF word 1–2-grams + logistic regression) on the same cases, then save everything.
from sklearn.feature_extraction.text import TfidfVectorizer
reference = {}
for name, chars in (("last_3000_chars", 3000), ("whole_text", None)):
    get = (lambda c: c["text"][-chars:]) if chars else (lambda c: c["text"])
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200000, sublinear_tf=True)
    Xtr = vec.fit_transform([get(c) for c in sample_train]); Xdv = vec.transform([get(c) for c in dev_cases])
    clf = LogisticRegression(max_iter=3000, C=4.0).fit(Xtr, y_train)
    reference[name] = round(best_macro_f1(y_dev, clf.predict_proba(Xdv)[:, 1]), 4)
    print(f"  TF-IDF + logistic regression, {name}: dev-clean Macro-F1 {reference[name]:.4f}")
json.dump({"health": health, "frozen_feature_probes": probe_results, "tfidf_reference": reference,
           "n_train_sample": len(train_ids), "n_dev_clean": len(dev_ids)},
          open(WORK_DIR / "diagnostic.json", "w"), indent=2)
print("Saved", WORK_DIR / "diagnostic.json")